# Notebook 4: Fusion Stage
### Simple comparison between Mistral vs Llama3 for multimodal fusion reasoning

### Fusion Modalities Input
1. Financial news sentiment analysis - in notebook 1
2. Chart/image pattern recognition - in notebook 2
3. Time-series forecasting - in notebook 3
4. Market positioning sentiment ratios (eg. EUR/USD -> 60% short, 40% long) - in *this* notebook

### Fusion Models Compared:
- Mistral (local Ollama deployment)
- Llama3 (local Ollama deployment)

### Step 0 - Basic check that we are in virtual environment

In [2]:
import sys
print(sys.executable)

C:\Users\KJ\Documents\School\FYP\Feature Prototype ENV\.venv\Scripts\python.exe


### Step 1 - Install dependencies and Imports

In [3]:
# Install dependencies
# %pip install ollama requests --quiet

In [19]:
# Imports
import json
import re
import ollama

### Step 2 - Hardcode all modality signals from models
- results from each modality's notebook will be manually pasted here, + 1 positional modality drawn from api and not from our used models
- in the full implementation, these flow will be automatic via a pipeline orchestrator

In [20]:
ASSET = "EURUSD"

# News/Sentiment (Notebook 1)
# Model selected: ProsusAI/finbert
news_signal = 1 # -1=bearish, 0=neutral, +1=bullish
news_conf = 0.76 # average confidence across headlines (should ideally be calculated beforehand then sent into fusion layer)
news_dist = {"positive": 3, "negative": 2, "neutral": 2} # signal distribution

# Chart/Image (Notebook 2)
# Model selected: foduucom YOLO
chart_signal = 1 # bullish (Head and shoulders bottom)
chart_conf = 0.619
chart_pattern = "Head and shoulders top"

# Time-Series (Notebook 3)
# Model selected: Custom GRU
# * all 3 models converged around ~56% directional accuracy, marginally above 50% random baseline
ts_signal = -1
ts_conf = 0.479
ts_accuracy = 55.95

# Positioning (rule-based, this notebook); in next cell

### Step 3 - Positioning modality (rule-based contrarian approach)
- unlike the 3 modalities signal generated from the models in previous 3 notebooks, this one draws data from api response
- for prototype we will just hardcode it to simulate MyFxBook community outlook API response
- full implementation fetches live from: https://www.myfxbook.com/api/get-community-outlook.json

In [21]:
positioning_raw = {
    "name": "EURUSD",
    "shortPercentage": 62,
    "longPercentage": 38,
}

# Contrarian rule - retail crowd tends to be wrong at extremes
# * >60% short -> contrarian bullish
# * >60% long -> contrarian bearish
# * otherwise -> neutral
CONTRARIAN_THRESHOLD = 60

short_pct = positioning_raw["shortPercentage"]
long_pct = positioning_raw["longPercentage"]

if short_pct >= CONTRARIAN_THRESHOLD:
    positioning_signal = 1
    positioning_conf   = round(short_pct / 100, 2)
elif long_pct >= CONTRARIAN_THRESHOLD:
    positioning_signal = -1
    positioning_conf = round(long_pct / 100, 2)
else:
    positioning_signal = 0
    positioning_conf = 0.5

print(f"Short%: {short_pct} | Long%: {long_pct}")
print(f"Positioning signal : {positioning_signal:+d}")
print(f"Positioning conf : {positioning_conf}")

Short%: 62 | Long%: 38
Positioning signal : +1
Positioning conf : 0.62


### Step 4 - Compute conflict level
- programmatically compute disagreement level across 4 modality signals before passing into fusion layer
- use simple rule-base consensus logic based off directional output (+1, -1, 0)
- raw modality outputs are still passed directly into the fusion prompt, conflict level only acts as an additional contextual feature

In [22]:
signals = {
    "news": news_signal,
    "chart": chart_signal,
    "timeseries": ts_signal,
    "positioning": positioning_signal
}

unique_signals = set(signals.values())
counts = {s: list(signals.values()).count(s) for s in unique_signals}
min_count = min(counts.values())

if len(unique_signals) == 1:
    conflict_level = "low"
    conflict_note = "all modalities agree"
elif len(unique_signals) == 2 and min_count == 1:
    conflict_level = "medium"
    conflict_note = "one modality disagrees with the other three"
elif len(unique_signals) == 2 and min_count == 2:
    conflict_level = "high"
    conflict_note = "modalities split evenly 2 vs 2"
else:
    conflict_level = "high"
    conflict_note = "modalities disagree significantly"

print("Pre-Fusion Signal Summary:")
for mod, sig in signals.items():
    direction = "bullish" if sig == 1 else "bearish" if sig == -1 else "neutral"
    print(f" {mod:<15} signal={sig:+d} ({direction})")
print(f"\nConflict level : {conflict_level} ({conflict_note})")

Pre-Fusion Signal Summary:
 news            signal=+1 (bullish)
 chart           signal=+1 (bullish)
 timeseries      signal=-1 (bearish)
 positioning     signal=+1 (bullish)

Conflict level : medium (one modality disagrees with the other three)


### Step 5 - Build fusion prompt
- passes only what the models actually output: labels, signals, confidence scores
- combines outputs from all 4 modalities into a single structured prompt
- there will be no hand-written summaries as LLM reasons from raw model outputs directly

In [23]:
def build_fusion_prompt(asset, signals,
                        news_conf, news_dist,
                        chart_conf, chart_pattern,
                        ts_conf, ts_accuracy,
                        positioning_conf, short_pct, long_pct,
                        conflict_level, conflict_note):
    return f"""You are a systematic forex market analyst for {asset}.
You will receive structured outputs from four independent analysis models.
Synthesise them into a final trading decision.

Return ONLY a valid JSON object. No explanation, no markdown, no code blocks. Raw JSON only.

MODALITY OUTPUTS:

1. News/Sentiment (ProsusAI/finbert)
   aggregate_signal: {signals['news']} (+1=bullish, -1=bearish, 0=neutral)
   average_confidence: {news_conf}
   signal_distribution: {news_dist['positive']} positive, {news_dist['negative']} negative, {news_dist['neutral']} neutral across 7 headlines

2. Chart Pattern (foduucom YOLO)
   signal: {signals['chart']} (+1=bullish, -1=bearish, 0=neutral)
   confidence: {chart_conf}
   detected_pattern: {chart_pattern}

3. Time-Series (Custom GRU)
   signal: {signals['timeseries']} (+1=bullish, -1=bearish, 0=neutral)
   confidence: {ts_conf}
   directional_accuracy: {ts_accuracy}%

4. Retail Positioning (contrarian rule-based)
   signal: {signals['positioning']} (+1=bullish, -1=bearish, 0=neutral)
   confidence: {positioning_conf}
   retail_short: {short_pct}%
   retail_long: {long_pct}%

Pre-computed conflict level: {conflict_level} ({conflict_note})

REQUIRED OUTPUT FORMAT:

{{
  "stance": "bullish" or "bearish" or "neutral",
  "confidence": <float 0.0 to 1.0>,
  "conflict_level": "low" or "medium" or "high",
  "signals": {{
    "news": "<one sentence based on the finbert output above>",
    "chart": "<one sentence based on the YOLO output above>",
    "timeseries": "<one sentence based on the GRU output above>",
    "positioning": "<one sentence based on the positioning data above>"
  }},
  "reasoning": "<2-3 sentences synthesising the overall stance, acknowledging any conflicts>"
}}"""

prompt = build_fusion_prompt(
    ASSET, signals,
    news_conf, news_dist,
    chart_conf, chart_pattern,
    ts_conf, ts_accuracy,
    positioning_conf, short_pct, long_pct,
    conflict_level, conflict_note
)

print("Fusion prompt built.")
print(f"Prompt length: {len(prompt)} chars\n")
print(prompt)

Fusion prompt built.
Prompt length: 1552 chars

You are a systematic forex market analyst for EURUSD.
You will receive structured outputs from four independent analysis models.
Synthesise them into a final trading decision.

Return ONLY a valid JSON object. No explanation, no markdown, no code blocks. Raw JSON only.

MODALITY OUTPUTS:

1. News/Sentiment (ProsusAI/finbert)
   aggregate_signal: 1 (+1=bullish, -1=bearish, 0=neutral)
   average_confidence: 0.76
   signal_distribution: 3 positive, 2 negative, 2 neutral across 7 headlines

2. Chart Pattern (foduucom YOLO)
   signal: 1 (+1=bullish, -1=bearish, 0=neutral)
   confidence: 0.619
   detected_pattern: Head and shoulders top

3. Time-Series (Custom GRU)
   signal: -1 (+1=bullish, -1=bearish, 0=neutral)
   confidence: 0.479
   directional_accuracy: 55.95%

4. Retail Positioning (contrarian rule-based)
   signal: 1 (+1=bullish, -1=bearish, 0=neutral)
   confidence: 0.62
   retail_short: 62%
   retail_long: 38%

Pre-computed conflict l

### Step 6 - extract and validate JSON from LLM response
- ensures LLM response are consistently and valid JSON
- in case of any edge case where LLM wrap output in format that is not JSON
- this will strip the code fence and extract raw JSON in a defensive manner

In [24]:
REQUIRED_KEYS = {"stance", "confidence", "conflict_level", "signals", "reasoning"}
REQUIRED_SIGNAL_KEYS = {"news", "chart", "timeseries", "positioning"}

def extract_and_validate(raw_text):
    text = re.sub(r"```json|```", "", raw_text).strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in LLM response")

    parsed = json.loads(text[start:end])

    issues = []
    missing_top = REQUIRED_KEYS - set(parsed.keys())
    missing_signals = REQUIRED_SIGNAL_KEYS - set(parsed.get("signals", {}).keys())
    if missing_top:
        issues.append(f"missing top-level keys: {missing_top}")
    if missing_signals:
        issues.append(f"missing signal keys: {missing_signals}")
    if parsed.get("stance") not in ["bullish", "bearish", "neutral"]:
        issues.append(f"unexpected stance value: {parsed.get('stance')}")
    if not isinstance(parsed.get("confidence"), (int, float)):
        issues.append("confidence is not a number")

    return parsed, issues

### Step 7 - Run fusion layer test with 2 LLM models
- Mistral and llama3 are selected due to them being lightweight, opensource and easily local-deployable

In [26]:
def run_fusion(model_name, prompt):
    print(f"Running fusion with {model_name}...\n")
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"temperature": 0.1} # low temp used for consistent structured output
    )
    raw = response["message"]["content"]
    print(f"Raw {model_name} Response:")
    print(raw)

    try:
        parsed, issues = extract_and_validate(raw)
        if issues:
            print(f"\nValidation issues: {issues}")
        else:
            print(f"\nJSON valid - all required keys present")
        return parsed
    except Exception as e:
        print(f"\nFailed to parse response: {e}")
        return None

mistral_output = run_fusion("mistral", prompt)
llama_output = run_fusion("llama3", prompt)

Running fusion with mistral...

Raw mistral Response:
{
  "stance": "bullish",
  "confidence": 0.648,
  "conflict_level": "medium",
  "signals": {
    "news": "News sentiment analysis indicates a bullish outlook with a majority of positive headlines.",
    "chart": "The Head and shoulders top pattern detected in the chart suggests a potential reversal to a bullish trend.",
    "timeseries": "Time-series analysis shows a bearish signal but with a relatively low confidence level.",
    "positioning": "Retail positioning data indicates that a contrarian approach would favor a long position, as retail traders are heavily short."
  },
  "reasoning": "The overall stance is bullish, despite some conflicting signals. The bullish news sentiment and chart pattern are supported by the contrarian retail positioning. However, it's important to note that the time-series analysis shows a bearish signal with lower confidence."
}

JSON valid - all required keys present
Running fusion with llama3...

Ra

### Step 8 - Comparison between both LLM models

In [27]:
print("Fusion Model Comparison:")
print(f"{'Criterion':<20} {'Mistral':<20} {'Llama3'}")
print("-" * 50)

if mistral_output and llama_output:
    print(f"{'Stance':<20} {mistral_output.get('stance','N/A'):<20} {llama_output.get('stance','N/A')}")
    print(f"{'Confidence':<20} {str(mistral_output.get('confidence','N/A')):<20} {str(llama_output.get('confidence','N/A'))}")
    print(f"{'Conflict level':<20} {mistral_output.get('conflict_level','N/A'):<20} {llama_output.get('conflict_level','N/A')}")
    print(f"{'Stance agreement?':<20} {'YES' if mistral_output.get('stance') == llama_output.get('stance') else 'NO'}")

print("\nMistral Reasoning:")
if mistral_output:
    print(mistral_output.get("reasoning", "N/A"))

print("\nLlama3 Reasoning:")
if llama_output:
    print(llama_output.get("reasoning", "N/A"))

Fusion Model Comparison:
Criterion            Mistral              Llama3
--------------------------------------------------
Stance               bullish              bullish
Confidence           0.648                0.76
Conflict level       medium               medium
Stance agreement?    YES

Mistral Reasoning:
The overall stance is bullish, despite some conflicting signals. The bullish news sentiment and chart pattern are supported by the contrarian retail positioning. However, it's important to note that the time-series analysis shows a bearish signal with lower confidence.

Llama3 Reasoning:
Despite some conflicting signals from the time-series model, the overall sentiment is bullish due to positive news and a head and shoulders top pattern. However, the positioning data suggests that retail traders may be overly bearish, which could create an opportunity for contrarian trading.


### Step 7 - Final observation + decision

Modality considerations:
- News/sentiment inputs currently use general financial headlines and are not filtered specifically for EURUSD-related events
- Chart/image detection confidence was moderate (~62%), so chart signals should still be interpreted cautiously
- Time-series directional accuracy remained relatively modest (~56%), so the forecasting modality should be treated as supportive rather than standalone
- Retail positioning uses a simplified contrarian rule-based interpretation for the prototype implementation

Result Observation:
- Both models generated valid JSON outputs with the required structure
- Both models also produced the same final bullish stance and medium conflict level
- Mistral gave slightly clearer reasoning when explaining how the different modality signals interacted
- Llama3 responses were more conversational and slightly more generic overall

Chosen model for pipeline and reasoning: **Mistral**
- Both models were able to perform multimodal fusion reasonably well for the prototype
- However, Mistral produced cleaner and more focused reasoning across the conflicting signals
- The responses also aligned better with the structured analytical style intended for the fusion layer

### EXTRA: Simulate feedback scoring mechanism
- this is the DESIGN ONLY for the feedback loop, not implementation for this prototype
- simulates how modality weight adjust EITHER based off user feedback OR automatically by itself
- in full implementation weight should persist across sessions and apply multiplier to confidence score before the fusion prompt

In [28]:
# Initial weight for all modality is flat 1.0, it will readjust over time depending on feedbakc mechanism
modality_weights = {
    "news": 1.0,
    "chart": 1.0,
    "timeseries": 1.0,
    "positioning": 1.0
}

# Simple example of a user feedback system where
def simulate_feedback(weights, outcome, signals, learning_rate=0.1):
    """
    outcome : +1 if user marks prediction correct, -1 if wrong
    modalities that agreed with final stance get boosted if correct,
    reduced if wrong. floor at 0.1 to avoid zeroing out a modality.
    """
    final_stance = 1 if sum(signals.values()) >= 0 else -1
    updated = weights.copy()
    for mod, sig in signals.items():
        agreed = (sig == final_stance)
        if outcome == 1 and agreed:
            updated[mod] = round(weights[mod] + learning_rate, 3)
        elif outcome == -1 and agreed:
            updated[mod] = round(max(0.1, weights[mod] - learning_rate), 3)
    return updated

print("Simulated Feedback Step (design only):")
print(f"Weights before : {modality_weights}")
updated_weights = simulate_feedback(modality_weights, outcome=1, signals=signals)
print(f"Weights after : {updated_weights}")
print("\nIn full implementation:")
print(" - weights persist to config file or database")
print(" - applied as multipliers to confidence scores before fusion prompt")
print(" - tracked over time to identify most reliable modalities")

Simulated Feedback Step (design only):
Weights before : {'news': 1.0, 'chart': 1.0, 'timeseries': 1.0, 'positioning': 1.0}
Weights after : {'news': 1.1, 'chart': 1.1, 'timeseries': 1.0, 'positioning': 1.1}

In full implementation:
 - weights persist to config file or database
 - applied as multipliers to confidence scores before fusion prompt
 - tracked over time to identify most reliable modalities
